In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# ===========================
# 函数：计算 30–70 hPa partial O3 column (DU)
# 与你之前的 calc_parc_o3 完全一致
# ===========================

def calc_parc_o3(var):
    """
    计算 30–70 hPa 的部分臭氧柱量 (DU)
    输入:
        var: xarray.DataArray, 维度包含 (time, plev/lev/level, lat, lon)
             值为 O3 混合比 (通常是体积分数 vmr)
    输出:
        O3: xarray.DataArray, 维度 (time, lat, lon)，单位 DU
    """
    m_air = 28.964 / (6.022E23)
    g = 980.6

    if "plev" in var.dims:
        plev = var.plev
        level = "plev"
    elif "lev" in var.dims:
        plev = var.lev
        level = "lev"
    else:
        plev = var.level
        level = "level"
        
    time = var.time
    delta_p = np.zeros((len(time), len(plev)))
    
    # 判断气压单位与排序
    if plev[0] < plev[len(plev)-1] and plev[len(plev)-1] <= 1000:
        factor = 100      # hPa -> Pa
        factor_2 = 1      # 30–70 hPa
    elif plev[0] > plev[len(plev)-1] and plev[0] <= 1000:
        factor = 100
        factor_2 = 1
    elif plev[0] < plev[len(plev)-1] and plev[len(plev)-1] > 1000:
        factor = 1        # 已是 Pa
        factor_2 = 100    # 3000–7000 Pa = 30–70 hPa
    else:
        factor = 1
        factor_2 = 100
    
    # 情况 1：气压自下而上递增
    if plev[0] < plev[len(plev)-1]:
        for levelx in range(1, len(plev)):
            delta_p[:, levelx].fill(plev[levelx] - plev[levelx-1])

        weights_p = xr.DataArray(
            delta_p * factor, dims=["time", level], coords=[time, plev]
        )
 
        O3 = var * weights_p * 10 / (g * m_air)
        
        if level == "plev":
            O3 = O3.sel(plev=slice(30*factor_2, 70*factor_2))
        elif level == "lev":
            O3 = O3.sel(lev=slice(30*factor_2, 70*factor_2))
        else:
            O3 = O3.sel(level=slice(30*factor_2, 70*factor_2))

        O3 = O3.sum(dim=level) / 2.687E16  # 转换为 DU
            
    # 情况 2：气压自下而上递减
    else:
        for levelx in range(0, len(plev)-1):
            delta_p[:, levelx].fill(plev[levelx] - plev[levelx+1])

        weights_p = xr.DataArray(
            delta_p * factor, dims=["time", level], coords=[time, plev]
        )
        O3 = var * weights_p * 10 / (g * m_air)
        
        if level == "plev":
            O3 = O3.sel(plev=slice(70*factor_2, 30*factor_2))
        elif level == "lev":
            O3 = O3.sel(lev=slice(70*factor_2, 30*factor_2))
        else:
            O3 = O3.sel(level=slice(70*factor_2, 30*factor_2))
                
        O3 = O3.sum(dim=level) / 2.687E16  # DU
    
    return O3.where(O3 != 0)


# ===========================
# 主程序：只读 0108 年 O3，并绘制该年的耗臭氧柱演变
# ===========================

# 0108 年 O3 文件路径
o3_file_0108 = (
    "/mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN/atm/hist/h1/o3/"
    "CO2x1SmidEmin_yBWCN.cam.h1.0108.O3.isobar.nc"
)

print("Reading:", o3_file_0108)
ds_0108 = xr.open_dataset(o3_file_0108)

# 取 O3 原始 4D 场
O3_4d = ds_0108["O3"]   # (time, plev/lev, lat, lon)

# 计算 30–70 hPa 部分柱 (DU)
O3_parc = calc_parc_o3(O3_4d)   # (time, lat, lon)

# 1) 经向平均 (zonal mean)
O3_zm = O3_parc.mean(dim="lon")   # (time, lat)

# 2) 选 60–90°N 极帽，并做 cosφ 加权平均
var_cap = O3_zm.sel(lat=slice(60, 90))
weights = np.cos(np.deg2rad(var_cap.lat))
var_cap_mean = var_cap.weighted(weights).mean(dim="lat")   # (time,)

# 3) 拿出 day-of-year 做 x 轴
doy = var_cap_mean["time"].dt.dayofyear.values
o3_series = var_cap_mean.values   # 对应每一天的 partial column (DU)

print("Data shape:", o3_series.shape, "(should be 365,)")

# ===========================
# 画图：0108 年 30–70 hPa, 60–90N O3 partial column (DU)
# ===========================

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(doy, o3_series, linewidth=2)

ax.set_xlim(1, 365)
ax.set_xlabel("Day of year (0108)", fontsize=14)
ax.set_ylabel("Partial ozone column 30–70 hPa (DU)", fontsize=14)
ax.set_title("Year 0108 O$_3$ partial column (30–70 hPa, 60–90°N)", fontsize=16)

ax.grid(True, alpha=0.3)

plt.tight_layout()

out_dir = "/home/weiji/test_code/plots"
os.makedirs(out_dir, exist_ok=True)
fig_path_png = os.path.join(out_dir, "O3_0108_partial_column_60N90N.png")
fig_path_pdf = os.path.join(out_dir, "O3_0108_partial_column_60N90N.pdf")

plt.savefig(fig_path_png, dpi=300)
plt.savefig(fig_path_pdf)

print("Saved figure to:")
print("  ", fig_path_png)
print("  ", fig_path_pdf)

plt.show()

# 关闭文件
ds_0108.close()
